### 1. Tabla de Hechos:  Fact_Segmentaciones_Dentales
**Granularidad:** 1 fila = 1 radiografía segmentada por la IA en una visita.

**Métricas numéricas puras:**
- `porcentaje_bacterias`: la métrica que pide el director (se promedia).
- `tiempo_inferencia_seg`: rendimiento del modelo de IA (sirve para monitoreo técnico).

**Llaves foráneas:**
- `id_paciente` → `Dim_Paciente`
- `id_sucursal` → `Dim_Sucursal`
- `id_tiempo` → `Dim_Tiempo`
- `id_radiografia` → `Dim_Radiografia`

`id_visita` se conserva como **dimensión degenerada** (identificador de la transacción, sin tabla propia).

### 2. Tablas de Dimensión

| Dimensión | Atributos | ¿Para qué sirve en la pregunta? |
|---|---|---|
| **Dim_Paciente** | `id_paciente`, `nombre`, `edad`, `rango_edad` (0-17, 18-35, 36-59, 60+) | Agrupar por **rango de edad** |
| **Dim_Sucursal** | `id_sucursal`, `nombre_sucursal`, `ciudad` | Agrupar por **sucursal** |
| **Dim_Tiempo** | `id_tiempo` (AAAAMMDD), `fecha`, `dia_semana`, `es_fin_de_semana`, `mes`, `anio` | Filtrar **fin de semana** y analizar **por mes** |
| **Dim_Radiografia** | `id_radiografia`, `ruta_mascara`, `hash_mascara` | Referencia a la imagen fuera del almacén (ver pregunta 3) |

`rango_edad` y `es_fin_de_semana` se precalculan en el ETL para que la consulta del director sea un simple filtro + groupby.

### 3. ETL: ¿Qué hacemos con la máscara Base64?

**Por qué sería catastrófico cargarla en la Tabla de Hechos:**
- **Tamaño:** 2 MB × 1 millón de visitas da igual a casi 2TB de texto en una tabla que debería pesar unos cuantos MB. La tabla de hechos se diseña para ser delgada y numérica.
- **Rendimiento:** cada groupby o escaneo tendría que leer esos strings aunque nadie los use. Las consultas pasarían de milisegundos a minutos, la RAM se satura y el caché se vuelve inútil.
- **No es analítico:** un string base64 no se suma, no se promedia y no se filtra. No aporta nada a un Roll-up.
- **Base64 infla el binario 33%:** además de pesado, es ineficiente como formato de almacenamiento.

**Cómo lo resuelvo en la tubería:**
1. **Extract:** leo la fila con la máscara.
2. **Transform:** decodifico el base64 a binario, lo guardo como archivo en un **object storage** (Azure Blob, S3, etc.) y calculo un **hash** para verificar integridad.
3. **Load:** en el almacén solo guardo la **ruta/URL** y el hash en `Dim_Radiografia`, y **elimino la columna base64** antes de cargar. La tabla de hechos solo recibe el `id_radiografia`.